# Model Training

Train the U-Net model for shadow boundary segmentation.
Includes loss comparison across different loss functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
import torch
from pathlib import Path
import sys

sys.path.insert(0, str(Path('..').resolve()))
from src.train import set_seed, build_model, build_dataloaders, train_one_epoch, validate, EarlyStopping
from src.models.losses import WeightedBCEDiceLoss

plt.rcParams['figure.dpi'] = 150

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

set_seed(config['project']['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Load Data and Build Model

In [ ]:
model = build_model(config, device)
train_loader, val_loader = build_dataloaders(config)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 2. Train with Weighted BCE + Dice Loss

In [ ]:
criterion = WeightedBCEDiceLoss(
    pos_weight=config['loss']['pos_weight'],
    dice_weight=config['loss']['dice_weight']
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config['training']['learning_rate'],
    weight_decay=config['training'].get('weight_decay', 1e-4)
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max',
    factor=config['training']['scheduler']['factor'],
    patience=config['training']['scheduler']['patience']
)

early_stopping = EarlyStopping(
    patience=config['training']['early_stopping']['patience'],
    metric='val_iou'
)

train_losses, val_losses = [], []
train_ious, val_ious = [], []
best_iou = 0.0
epochs = min(50, config['training']['epochs'])

for epoch in range(epochs):
    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = validate(model, val_loader, criterion, device)
    
    scheduler.step(val_metrics['iou'])
    
    train_losses.append(train_metrics['loss'])
    val_losses.append(val_metrics['loss'])
    train_ious.append(train_metrics['iou'])
    val_ious.append(val_metrics['iou'])
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | "
              f"Train Loss: {train_metrics['loss']:.4f} IoU: {train_metrics['iou']:.4f} | "
              f"Val IoU: {val_metrics['iou']:.4f} Dice: {val_metrics['dice']:.4f}")
    
    if val_metrics['iou'] > best_iou:
        best_iou = val_metrics['iou']
        torch.save(model.state_dict(), '../outputs/checkpoints/best_model.pth')
    
    if early_stopping(val_metrics['iou']):
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest validation IoU: {best_iou:.4f}")

## 3. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses, label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_ious, label='Train')
axes[1].plot(val_ious, label='Val')
axes[1].set_title('IoU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()